# 3-Trainig the models 


**Entraînement et comparaison des modèles** — trois modèles de classification (Random Forest, Logistic Regression, SVM) sont entraînés sur `X_train_prepared` / `y_train`, puis évalués sur le jeu de test indépendant avec recall, precision, F1 Score, Accuracy et ROC AUC (moyenne pondérée, cible à 3 classes). Le résultat est synthétisé dans le tableau `results` pour identifier le meilleur modèle.

In [ ]:
from pathlib import Path

import joblib

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


DATA_PROCESSED_DIR = (Path.cwd() / "Data" / "processed").resolve()
print(DATA_PROCESSED_DIR)


split = joblib.load(DATA_PROCESSED_DIR / "split_data.pkl")
X_train, X_test = split["X_train"], split["X_test"]
y_train, y_test = split["y_train"], split["y_test"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

MODELS = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

rows = []
for model_name, model in MODELS.items():
    pipeline = Pipeline([
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)

    recall = recall_score(y_test, predictions, average="weighted")
    precision = precision_score(y_test, predictions, average="weighted")
    f1 = f1_score(y_test, predictions, average="weighted")
    accuracy = accuracy_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")

    print(f"{model_name} Results:")
    print(f"Recall:    {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"ROC AUC:   {roc_auc:.4f}")
    print("-" * 30)

    rows.append({
        "Model": model_name,
        "Recall": recall,
        "Precision": precision,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "ROC AUC": roc_auc,
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))


/home/bahaztariq/Youcode/mental/src/mental/Data/processed
X_train: (1600, 21) | X_test: (400, 21)
y_train: (1600,) | y_test: (400,)


Random Forest Results:
Recall:    0.6775
Precision: 0.6696
F1 Score:  0.6721
Accuracy:  0.6775
ROC AUC:   0.8528
------------------------------
Logistic Regression Results:
Recall:    0.7575
Precision: 0.7576
F1 Score:  0.7575
Accuracy:  0.7575
ROC AUC:   0.9150
------------------------------


/home/bahaztariq/Youcode/mental/.venv/lib/python3.15/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM Results:
Recall:    0.7225
Precision: 0.7205
F1 Score:  0.7214
Accuracy:  0.7225
ROC AUC:   0.8944
------------------------------
              Model  Recall  Precision  F1 Score  Accuracy  ROC AUC
      Random Forest  0.6775   0.669641  0.672101    0.6775 0.852769
Logistic Regression  0.7575   0.757557  0.757498    0.7575 0.915043
                SVM  0.7225   0.720469  0.721403    0.7225 0.894395


**Sauvegarde du meilleur modèle** — le modèle avec le meilleur F1 Score est ré-entraîné puis sérialisé en `.pkl` (joblib) pour l'étape 4 (optimisation).

In [2]:
import joblib

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Meilleur modèle = plus grand F1 Score dans le tableau `results`
best_model_name = results.loc[results["F1 Score"].idxmax(), "Model"]
print(f"Meilleur modèle : {best_model_name}")

# Ré-entraînement du seul meilleur modèle puis sauvegarde en joblib
best_pipeline = Pipeline([("classifier", MODELS[best_model_name])])
best_pipeline.fit(X_train, y_train)

out_path = MODELS_DIR / f"{best_model_name.lower().replace(' ', '_')}_baseline.pkl"
joblib.dump(best_pipeline, out_path)
print(f"Sauvegardé : {out_path} ({out_path.stat().st_size / 1024:.1f} Ko)")


Meilleur modèle : Logistic Regression
Sauvegardé : /home/bahaztariq/Youcode/mental/src/mental/models/logistic_regression_baseline.pkl (1.6 Ko)
